In [6]:
import numpy as np
import pandas as pd
from sqlalchemy import create_engine

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)



# --- Create connection string ---
DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)


#Test Query for customers table
query_customers = "SELECT * FROM customers;"
df_customers = pd.read_sql_query(query_customers, engine)
'''
print("Data loaded successfully.")
print(df_customers.head(10))
print(f"Data shape: {df_customers.shape}")
print(df_customers.info())

# Data cleaning for customers
# Keep only alphabets and spaces in name column
df_customers['name'] = df_customers['name'].astype(str).str.replace(r'[^a-zA-Z\s]', '', regex=True)
# keep only alphabets in city column
df_customers['city'] = df_customers['city'].astype(str).str.replace(r'[^a-zA-Z]', '', regex=True)
# keep only numbers in phone and age columns
for col in ['phone', 'age']:
    df_customers[col] = df_customers[col].astype(str).str.replace(r'[^0-9]', '', regex=True)
    df_customers[col] = pd.to_numeric(df_customers[col], errors='coerce').astype('Int64')

#Extract a list of unique values from name, city, phone and age columns
unique_values = {
    'name': df_customers['name'].unique(),
    'city': df_customers['city'].unique(),
    'phone': df_customers['phone'].unique(),
    'age': df_customers['age'].unique()
}

# show list of unique values
for col, values in unique_values.items():
    print(f"Unique values in {col}: {values}")
# corrections for coulumns
corrections = {
    'Khalid Mehmood': ['Talagang', 34, 3125675430],
    'Maryam Javed': ['Tehi', 28, 3085673216],
    'Rabia Aslam': ['Malakwal', 32, 3297649876],
    'Nida Farooq': ['Akwal', 26, 3033412089],
    'Usman Tariq': ['Jhatla', 30, 3008725676],
    'Sara Malik': ['Kotsarang', 27, 3059934709],
    'Zara Sheikh': ['Talagang', 29, 3089356732],
    'Bilal Hussain': ['Murali', 31, 3009655342],
    'Fatima Noor': ['Rehmanabad', 25, 3089045670],
    'Hamza Raza': ['Tehi', 28, 3446732189],
    'Hina Rashid': ['Talagang', 32, 3345632198],
    'Shahid Anwar': ['Kotsarang', 33, 3112785432],
    'Nimra Rehman': ['Akwal', 24, 3149943211],
    'Ayesha Siddiqui': ['Chinji', 29, 3096329153],
    'Hassan Ali': ['Talagang', 27, 3145678901],
    'Ahmed Khan': ['Malakwal', 30, 3121986567],
    'Adnan Iqbal': ['Chinji', 26, 3096432901],
    'Imran Qureshi': ['Talagang', 28, 3088892761],
    'Tariq Mahmood': ['Kotsarang', 31, 3179935978],
    'Sana Shamim': ['Jhatla', 25, 319076213]
}
for name, (city, age, phone) in corrections.items():
    df_customers.loc[df_customers['name'] == name, ['city', 'age', 'phone']] = [city, age, phone]
# drop duplicates from name column
df_customers = df_customers.drop_duplicates(subset=['name'])
# rearange customer_id from 1 to onward
df_customers = df_customers.reset_index(drop=True)
df_customers['customer_id'] = df_customers.index + 1

# Export cleaned data to a csv file
df_customers.to_csv('customers_cleaned.csv', index=False)

# create a new table customers_cleaned and insert into database
df_customers.to_sql('customers_cleaned', engine, if_exists='replace', index=False)
'''

#Test query for products table
query_products = "SELECT * FROM products;"
df_products = pd.read_sql_query(query_products, engine)
'''
print("Data loaded successfully.")
print(df_products.head(10))
print(f"Data shape: {df_products.shape}")
print(df_products.info())


# --- CLEAN PRODUCT NAME ---
df_products['product_name'] = (
    df_products['product_name']
    .astype(str)
    .str.replace(r'[^a-zA-Z0-9\s\.]', '', regex=True)   # keep letters, digits, spaces, dot, hyphen
    .str.replace(r'\s+', ' ', regex=True)                 # remove multiple spaces
    .str.strip()                                          # remove leading/trailing spaces
)

# --- CLEAN CATEGORY (only alphabets + space) ---
df_products['category'] = (
    df_products['category']
    .astype(str)
    .str.replace(r'[^a-zA-Z\s]', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

# --- CLEAN COST & RETAIL PRICE ---
for col in ['cost_price', 'retail_price']:
    df_products[col] = (
        df_products[col]
        .astype(str)
        .str.replace(r'[^0-9\.]', '', regex=True)   # keep numbers + decimal
        .str.replace(r'\.+', '.', regex=True)       # fix multiple decimals
        .str.strip()
    )

    # Convert to numeric safely
    df_products[col] = pd.to_numeric(df_products[col], errors='coerce')

# list of values from product_name
unique_product_names = df_products['product_name'].unique()
print(f"Unique product names: {unique_product_names}")

product_category_map = {
    # -------------------------
    # GROCERY
    # -------------------------
    'Nimko Mix 400g': 'Grocery',
    'Sufi Banaspati 1kg': 'Grocery',
    'Tea Whitener 1kg': 'Grocery',
    'National Ketchup 1kg': 'Grocery',
    'Shan Biryani Masala': 'Grocery',
    'Rice Basmati 5kg': 'Grocery',
    'Wheat Flour 10kg': 'Grocery',
    'Olpers Milk 1L': 'Grocery',
    'Nestle Milk Pack 1L': 'Grocery',
    'Tapal Danedar 950g': 'Grocery',
    'Dalda Cooking Oil 1L': 'Grocery',

    # -------------------------
    # BEVERAGES
    # -------------------------
    'Sprite 1.5L': 'Beverages',
    'Pepsi 1.5L': 'Beverages',
    'Coca Cola 1.5L': 'Beverages',

    # -------------------------
    # HOUSEHOLD
    # -------------------------
    'Ariel Washing Powder 1kg': 'Household',
    'Surf Excel 1kg': 'Household',
    'Tissue Roll Rose Petal': 'Household',

    # -------------------------
    # PERSONAL CARE
    # -------------------------
    'Dove Shampoo 360ml': 'Personal Care',
    'Dettol Handwash 200ml': 'Personal Care',
    'Colgate Toothpaste 100ml': 'Personal Care',
    'Lux Soap 110g': 'Personal Care',
    'Lifebuoy Soap 110g': 'Personal Care',

    # -------------------------
    # OTHER
    # -------------------------
    'None': 'Other'
}

# Map product names to categories
df_products['category'] = df_products['product_name'].map(product_category_map)

# remove duplicate product names
df_products = df_products.drop_duplicates(subset=['product_name'])
# rearange product_id from 1 to onward
df_products = df_products.reset_index(drop=True)
df_products['product_id'] = df_products.index + 1

# create a new table products_cleaned and insert into database
df_products.to_sql('products_cleaned', engine, if_exists='replace', index=False)
# Export cleaned data to a csv file
df_products.to_csv('products_cleaned.csv', index=False)'''



# --- Test query for sales table ---
query_sales = "SELECT * FROM sales;"
df_sales = pd.read_sql_query(query_sales, engine)

print("Data loaded successfully.")
print(df_sales.head(10))
print(f"Data shape: {df_sales.shape}")
print(df_sales.info())

# --- Data Cleaning for sales ---
# Removing special characters from product_id, customer_id, quantity and keep only numbers
for col in ['product_id', 'customer_id', 'quantity']:
    df_sales[col] = df_sales[col].astype(str).str.replace(r'[^0-9]', '', regex=True)
    df_sales[col] = pd.to_numeric(df_sales[col], errors='coerce').astype('Int64')
# Removing special characters from price and keep only numbers and decimal point
df_sales['price'] = df_sales['price'].astype(str).str.replace(r'[^0-9\.]', '', regex=True)
df_sales['price'] = pd.to_numeric(df_sales['price'], errors='coerce')
# Convert date column to datetime format
df_sales['date'] = pd.to_datetime(df_sales['date'], errors='coerce')
#(Data types have already been changed in the above steps)


#-------------------Attempting--------------
query_products_cleaned = "SELECT * FROM products_cleaned;"
df_products_cleaned = pd.read_sql_query(query_products_cleaned, engine)

query_sales_cleaned = "SELECT * FROM sales_cleaned;"
df_sales_cleaned = pd.read_sql_query(query_sales_cleaned, engine)

# STEP 1: Map product_name into sales (using raw product table)
df_sales_cleaned = df_sales_cleaned.merge(
    df_products[['product_id', 'product_name']],
    on='product_id',
    how='left'
)

# STEP 2: Map each product_name to its clean product_id + retail_price
products_map = df_products_cleaned[['product_id', 'product_name', 'retail_price']]

df_sales_cleaned = df_sales_cleaned.merge(
    products_map,
    on='product_name',
    how='left',
    suffixes=('_old', '')
)

# STEP 3: Replace old product_id with cleaned one
df_sales_cleaned['product_id'] = df_sales_cleaned['product_id']

# STEP 4: Replace the sales price with the retail_price from products_cleaned
df_sales_cleaned['price'] = df_sales_cleaned['retail_price']

# STEP 5: Drop helper columns
df_sales_cleaned = df_sales_cleaned.drop(columns=['product_id_old', 'retail_price'])

# Replace customer_id with random numbers between 1 and 20
df_sales_cleaned['customer_id'] = np.random.randint(1, 21, size=len(df_sales_cleaned))
# Keep only alphabets, numbers, decimals in product_name, remove extra spaces
df_sales_cleaned['product_name'] = (
    df_sales_cleaned['product_name']
    .astype(str)
    .str.replace(r'[^a-zA-Z0-9\s\.]', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

# Covert product_id and quantity to integer

for col in ['product_id', 'quantity']:
    df_sales_cleaned[col] = pd.to_numeric(df_sales_cleaned[col], errors='coerce').astype('Int64')

# Keep dates only from 2025-08-01 to 2025-11-15
df_sales_cleaned = df_sales_cleaned[
    (df_sales_cleaned['date'] >= '2025-08-01') &
    (df_sales_cleaned['date'] <= '2025-11-15')
]
# Re_arranging count
df_Sales_3 = (
    df_sales_cleaned
    .groupby('date', group_keys=False)
    .apply(lambda g: g.sample(
        n=np.random.randint(50, min(112, len(g)) + 1)
    ))
    .reset_index(drop=True)
)

# Re arrange sales_id from 1 to onward
df_Sales_3 = df_Sales_3.reset_index(drop=True)
df_Sales_3['sale_id'] = df_Sales_3.index + 1

# Create a new csv file for cleaned sales data
df_Sales_3.to_csv('sales_cleaned_3.csv', index=False)

# Create a new table sales_cleaned_3 and insert into database
df_Sales_3.to_sql('sales_cleaned_3', engine, if_exists='replace', index=False)






<>:33: SyntaxWarning: invalid escape sequence '\s'
<>:104: SyntaxWarning: invalid escape sequence '\s'
<>:33: SyntaxWarning: invalid escape sequence '\s'
<>:104: SyntaxWarning: invalid escape sequence '\s'
C:\Users\Al-Abbas Home PC\AppData\Local\Temp\ipykernel_16248\2406378989.py:33: SyntaxWarning: invalid escape sequence '\s'
  df_customers['name'] = df_customers['name'].astype(str).str.replace(r'[^a-zA-Z\s]', '', regex=True)
C:\Users\Al-Abbas Home PC\AppData\Local\Temp\ipykernel_16248\2406378989.py:104: SyntaxWarning: invalid escape sequence '\s'
  .str.replace(r'[^a-zA-Z0-9\s\.]', '', regex=True)   # keep letters, digits, spaces, dot, hyphen


Data loaded successfully.
   sale_id product_id customer_id quantity   price        date
0        1         84        1077        2  468.69  2025-05-19
1        2         92        1243        3  261.06  2025-05-19
2        3        449      2156??        5   944.0  2025-05-19
3        4         98         18         1   332.9  2025-05-19
4        5       None         911        1  None??  2025-05-19
5        6        203        2567        1  897.73  2025-05-19
6        7        143        2488        1  181.27  2025-05-19
7        8        177       1836         4  652.82  2025-05-19
8        9        141        1812        2  602.55  2025-05-19
9       10         43        None        1  152.82  2025-05-19
Data shape: (20000, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   sale_id      20000 non-null  int64 
 1   product_id   19053 non

C:\Users\Al-Abbas Home PC\AppData\Local\Temp\ipykernel_16248\2406378989.py:276: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(


321